In [0]:
schemalocation = "/Volumes/ev_spark/myvol/checkpoint/schema/bc_ev/"
sourcepath = "/Volumes/ev_spark/myvol/landing/bc_ev_pop/"
chkptpath = "/Volumes/ev_spark/myvol/checkpoint/chkpt/bc_ev/"

## Method to read BC EV pop from source file

In [0]:
def readRawBCEV_df():
    from pyspark.sql.functions import current_timestamp
    from pyspark.sql.types import StructType, StructField, StringType

    # Define explicit schema to override cached incorrect schema
    schema = StructType([
        StructField("city", StringType(), True),
        StructField("year_2021", StringType(), True),
        StructField("year_2022", StringType(), True),
        StructField("year_2023", StringType(), True),
        StructField("year_2024", StringType(), True),
        StructField("year_2025", StringType(), True)
    ])

    readRawBCEV_df = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", "csv")
                        .option("cloudFiles.schemaLocation", schemalocation)
                        .option("header", "true")
                        .schema(schema)
                        .load(sourcepath)
                        .withColumn("extraction_date", current_timestamp())
                        
                    )
    return readRawBCEV_df

## Method to write read BC EV pop to delta table

In [0]:
def writeBCEV_df(df):
    (df.writeStream
        .format("delta")
        .option("checkpointLocation", chkptpath)
        .outputMode("append")
        .trigger(availableNow= True)
        .toTable("ev_spark.bronze.bc_ev")
    )

## Calling methods to read and write BC EV pop

In [0]:
read_df = readRawBCEV_df()
writeBCEV_df(read_df)

In [0]:
%sql
select * from ev_spark.bronze.bc_ev